- Identify duplicate transactions using transaction_id.
- Keep the latest version of a transaction based on transaction_ts.
- If two records have the same transaction_id and same transaction_ts, use ingestion_timestamp as the tie-breaker.
- Produce one record per transaction_id.

In [0]:
%python
from pyspark.sql import functions as F

from pyspark.sql.window import Window


BRONZE_PATH = "/mnt/usvikformula1dl/bronze"
transaction_bronze_path = f"{BRONZE_PATH}/transaction_fact"
SILVER_PATH = "/mnt/usvikformula1dl/silver"
transaction_df=spark.read.format("delta").load(f"{transaction_bronze_path}")

window_spec=Window().partitionBy("transaction_id").orderBy(F.col("transaction_ts").desc())
transaction_df_final=transaction_df.withColumn("rownum", F.row_number().over(window_spec)).filter(F.col("rownum")==1).drop("rownum")
transaction_df_final.write.mode("overwrite").format("delta").save(f"{SILVER_PATH}/transaction_fact")
test_df=spark.read.format("delta").load(f"{SILVER_PATH}/transaction_fact")
display(test_df)



In [0]:
%python
transaction_df_final.groupBy("transaction_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()